In [ ]:
from __future__ import annotations
from typing import List, Tuple
import numpy as np
import os
from tqdm import tqdm, trange

from collections import defaultdict
import pandas as pd


import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class PatientDataset():
    """
    A dataset container for time-series patient medical records with sepsis labels.

    Attributes:
        data (List[Tuple[np.ndarray, np.ndarray]]): List of patient records, where each
            element is a tuple (x, y):
            - x: np.ndarray of shape (n_timesteps, n_features) - feature matrix for one patient
            - y: np.ndarray of shape (n_timesteps,) - binary sepsis labels (0/1) for each timestep
        feature_names (List[str]): Names of the clinical features in the same order as
            they appear in the feature dimension of x.

    Data Structure:
        - Each patient can have a different number of timesteps (n_timesteps varies)
        - All patients must have the same number of features (n_features is constant)
        - Features are stored as numpy arrays with shape (n_timesteps, n_features)
        - Labels are binary (0 = no sepsis, 1 = sepsis) with shape (n_timesteps,)

    Methods:
        __len__(): Returns the total number of patients in the dataset
        __getitem__(idx): Returns the (x, y) tuple for the patient at index idx
        from_file(file_name, subset): Class method to load dataset from .npz file

    Example:
        >>> # Load from file
        >>> dataset = PatientDataset.from_file('train.npz')
        >>> print(f"Total patients: {len(dataset)}")
        >>> print(f"Features: {dataset.feature_names}")
        >>>
        >>> # Access individual patient data
        >>> x, y = dataset[0]
        >>> print(f"Patient 0: {x.shape[0]} timesteps, {x.shape[1]} features")
        >>>
        >>> # Iterate over all patients
        >>> for x, y in dataset:
        ...     # Process each patient's time series
        ...     pass
    """

    def __init__(
        self,
        data: List[Tuple[np.ndarray, np.ndarray]],
        feature_names: List[str],
    ) -> None:

        self.data = data
        self.feature_names = feature_names

        # ensure that all patients have the same number of features
        n_features = len(self.feature_names)
        for x, _ in self.data:
            assert x.shape[1] == n_features, "Number of features do not match across patients."

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Tuple[np.ndarray, np.ndarray]:
        """
        Returns data for a single patient.
        There are `n_sample` timesteps for which data & sepsis labels are available.
        x: np.ndarray of shape (n_samples, n_features)
        y: np.ndarray of shape (n_samples,)
        """
        return self.data[idx]

    @classmethod
    def from_file(cls, file_name: str, subset = None) -> PatientDataset:
        data = np.load(file_name)
        x_data = data['x']
        y_data = data['y']
        patient_to_num_records = data['patient_to_num_records']
        feature_names = data['feature_names'].tolist()

        # unzip the data
        unzipped_data = []
        start = 0
        for c, num_records in enumerate(patient_to_num_records):
            if subset is not None and c >= subset:
                break

            end = start + num_records
            x = x_data[start:end]
            y = y_data[start:end]
            unzipped_data.append((x, y))
            start = end

        return cls(data=unzipped_data, feature_names=feature_names)


# 1 Exploring the data

Run the following cell to load in the training and validation datasets. Both datasets are instances of `PatientDataset`. Please see the documentation above for information about how to use a `PatientDataset`.

In [ ]:
# Load in datasets. 
dataset_trn = PatientDataset.from_file("trn.npz")
dataset_val = PatientDataset.from_file("val.npz")
feature_names = dataset_trn.feature_names

# We will fill in the missing data in this section, but need a copy of the raw
# data with missing values for next section.
dataset_trn_raw = dataset_trn
dataset_val_raw = dataset_val

## 1(a) Dataset sizes

In [ ]:
# TODO: 1(a) Dataset sizes.

...

## 1(b) Lengths of patient records

In [ ]:
# TODO: 1(b) Lengths of patient records.

...

## 1(c) Percentage of sepsis patients

In [ ]:
# TODO: 1(c) Percentage of sepsis patients.

...

## 1(d) Reversed labels

In [ ]:
# TODO: 1(d) Reversed labels.

...

## 1(e) Age distributions

In [ ]:
# TODO: 1(e) Age distributions.

...

## 1(f) Handling missing data

The cell below computes the vector $\boldsymbol{\mu} \in \mathbb{R}^{40}$ of the training-set means of each of the features. 

In [ ]:
def compute_means(dataset=dataset_trn):
    # Calculate mu_d using the first timestep of each patient
    x_first = np.stack([x[0] for x, _ in dataset])

    means = np.nanmean(x_first, axis=0)
    means = np.nan_to_num(means, nan=0.0)

    means = dict(zip(dataset.feature_names, means))
    return means

mus = compute_means()

In [ ]:
# TODO: 1(f)(i) Implement the filling rule. 

def forward_fill_with_mean(dataset, means = mus):
    filled_dataset = []

    for x, y in tqdm(dataset, desc="Forward-fill imputing dataset"):
        x_filled = x.copy()

        ###
        # TODO: 1(f)(i). Fill in the missing entries of x_filled according to the rule.

        ###

        filled_dataset.append((x_filled, y))

    return filled_dataset

In [ ]:
dataset_trn = forward_fill_with_mean(dataset_trn)
dataset_val = forward_fill_with_mean(dataset_val)

In [ ]:
# TODO: 1(f)(ii) Report the mean heart rate on the training set after filling.

...

# 2 Evaluating model performance

Implementations of GHA's utility functions are provided in the cell below.

In [ ]:
def patient_utility(predictions, labels):
    """
    Returns the utility of a method's predictions on a single patient.

    predictions: np.ndarray with 0,1 predictions per timestep (n_timesteps,)
    labels: np.ndarray with 0,1 labels per timestep (n_timesteps,)
    returns: utility value
    """
    dt_early = -12
    dt_optimal = -6
    dt_late = 3.0
    max_u_tp = 1
    min_u_fn = -2
    u_fp = -0.05
    u_tn = 0

    assert len(predictions) == len(labels), "Number of predictions and labels do not match."

    # Does the patient eventually have sepsis?
    if np.any(labels):
        is_septic = True
        t_sepsis = np.argmax(labels) - dt_optimal
    else:
        is_septic = False
        t_sepsis = float('inf')

    n = len(labels)

    # Define slopes and intercept points for utility functions of the form
    # u = m * t + b.
    m_1 = float(max_u_tp) / float(dt_optimal - dt_early)
    b_1 = -m_1 * dt_early
    m_2 = float(-max_u_tp) / float(dt_late - dt_optimal)
    b_2 = -m_2 * dt_late
    m_3 = float(min_u_fn) / float(dt_late - dt_optimal)
    b_3 = -m_3 * dt_optimal

    # Compare predicted and true conditions.
    u = np.zeros(n)
    for t in range(n):
        if t <= t_sepsis + dt_late:
            # TP
            if is_septic and predictions[t]:
                if t <= t_sepsis + dt_optimal:
                    u[t] = max(m_1 * (t - t_sepsis) + b_1, u_fp)
                elif t <= t_sepsis + dt_late:
                    u[t] = m_2 * (t - t_sepsis) + b_2
            # FP
            elif not is_septic and predictions[t]:
                u[t] = u_fp
            # FN
            elif is_septic and not predictions[t]:
                if t <= t_sepsis + dt_optimal:
                    u[t] = 0
                elif t <= t_sepsis + dt_late:
                    u[t] = m_3 * (t - t_sepsis) + b_3
            # TN
            elif not is_septic and not predictions[t]:
                u[t] = u_tn

    # Find total utility for patient.
    return np.sum(u).item()

def compute_utility(predictions_list, dataset, threshold=None):
    """
    Computes normalized utility from pre-computed predictions or probabilities.
    Normalization: (observed - zero) / (best - zero)

    Args:
        predictions_list: List of np.ndarray predictions (binary) or probabilities for each patient
        utility_fn: UtilityFunction instance
        threshold: Optional threshold to convert probabilities to binary predictions
        dataset: Optional dataset (List of (x, y) tuples) to extract labels from

    Returns:
        If threshold is provided: dict with 'utility' and 'threshold' keys
        Otherwise: normalized utility value
    """
    labels_list = [y for x, y in dataset]

    # Convert probabilities to binary predictions if threshold is provided
    if threshold is not None:
        predictions_list = [(probs >= threshold).astype(int) for probs in predictions_list]

    dt_early = -12
    dt_optimal = -6
    dt_late = 3

    num_patients = len(predictions_list)
    obsr_utilities = np.zeros(num_patients)
    orac_utilities = np.zeros(num_patients)
    zero_utilities = np.zeros(num_patients)

    for k, (preds, labels) in enumerate(zip(predictions_list, labels_list)):
        if torch.is_tensor(preds):
            preds = preds.cpu().numpy()
        preds = np.array(preds)

        num_rows = len(labels)

        # Oracle predictions: predict in the optimal window
        orac_predictions = np.zeros(num_rows)
        if np.any(labels):
            t_sepsis = np.argmax(labels) - dt_optimal
            orac_predictions[max(0, t_sepsis + dt_early):min(t_sepsis + dt_late + 1, num_rows)] = 1

        zero_predictions = np.zeros(num_rows)

        obsr_utilities[k] = patient_utility(preds, labels)
        orac_utilities[k] = patient_utility(orac_predictions, labels)
        zero_utilities[k] = patient_utility(zero_predictions, labels)

    # Compute normalized utility
    obsr_utility = np.sum(obsr_utilities)
    orac_utility = np.sum(orac_utilities)
    zero_utility = np.sum(zero_utilities)

    utility = (obsr_utility - zero_utility) / (orac_utility - zero_utility)

    return utility

# 3 Baselines

## 3(a) Never-sepsis baseline

In [ ]:
# TODO: 3(a) Never-sepsis baseline.

...

## 3(b) Always-sepsis baseline

In [ ]:
# TODO: 3(b) Always-sepsis baseline.

...

## 3(c) Logistic regression

### 3(c)(i) Missingness logistic regression

In [ ]:
# TODO: 3(c)(i) Missingness logistic regression.

class MissingnessLogisticModel:
    def __init__(self):
        self.model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)

    @staticmethod
    def indicators(x):
        # TODO: 3(c)(i). Return the binary missingness vector for a feature vector x.
        
        return ...

    def fit(self, dataset):
        # TODO: 3(c)(i). From the dataset, create a new dataset where labels are the 
        # sepsis labels and indicators are the missingness vectors.

        self.model.fit(indicators, labels)
        return self

    def predict(self, x):
        return self.model.predict_proba(self.indicators(x))[:, 1]

### 3(c)(ii) Implement threshold optimization

In [ ]:
# TODO: 3(c)(ii) Implement threshold optimization.

def optimize_threshold(predicted_probs_list, dataset, thresholds=None):
    """
    Optimize threshold using provided probability predictions and normalized utility.

    Args:
        predicted_probs_list: List of np.ndarray with predicted probabilities for each patient
        dataset: List of (x, y) tuples for labels
        utility_fn: UtilityFunction instance
        thresholds: Array of thresholds to test
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.2, 20)

    threshold_utilities = {}

    ###
    # TODO: 3(c)(ii). Compute the best threshold and its utility on dataset. Use 
    # compute_utility to calculate utilities.
    ...
    best_threshold = ...
    best_utility = ...
    ###

    return {
        'best_threshold': best_threshold,
        'best_utility': best_utility
    }

### 3(c)(iii) Threshold selection for logistic regression

In [ ]:
# TODO: 3(c)(iii) Threshold selection for logistic regression

# Train the logistic regression on dataset_trn_raw.
...

# Select the optimal threshold on dataset_val_raw.
...

# 4 Attention

### 4(a)(ii), 4(b)(iii), 4(c)(i), 4(d)(i) Implement the transformer

In [ ]:
# TODO: 4(a)(ii), 4(b)(iii), 4(c)(i), 4(d)(i) Implement the transformer.

class Transformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=3, dim_feedforward=256):
        super().__init__()

        ###
        # TODO: 4(a)(ii). Implement the input layer.
        self.input_layer = ...
        ### 

        # Provided encoder layer: attention and feedforward are already implemented.
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            activation="relu",
            dropout=0.0,
            batch_first=True,
        )

        ###
        # TODO: 4(b)(iii). Implement the encoder.
        self.transformer_encoder = ...
        ###

        ### 
        # TODO: 4(c)(i). Implement the outer layer.
        self.output_layer = ...
        ###

    def forward(self, x, attention_mask=None):
        padding_mask = ~attention_mask.bool() if attention_mask is not None else None
        ###
        # TODO: 4(d)(i). Implement the forward pass. Pass in padding_mask to the 
        # src_key_padding_mask input when calling self.transformer_encoder.
        ...
        return ...
        ###

### 4(d)(ii) Train the transformer

Elvi's code for training the transformer is provided below. 

In [ ]:
def get_transformer_predictions(transformer_model, dataset, device):
    """
    Generate predicted probabilities for each patient in the dataset.
    Uses optimized batch prediction - processes full sequence in one forward pass.
    """
    transformer_model.eval()
    predictions_list = []

    for x, y in tqdm(dataset, desc="Generating transformer predictions"):
        # Convert to tensor and add batch dimension
        x_tensor = torch.FloatTensor(x).unsqueeze(0).to(device)  # (1, T, n_features)

        # Create attention mask (all True since this is a real sequence)
        attention_mask = torch.ones((1, x.shape[0]), dtype=torch.bool).to(device)

        with torch.no_grad():
            # Process full sequence in one forward pass
            outputs = transformer_model(x_tensor, attention_mask)  # (1, T, 2)
            # Get probabilities for all time steps
            probs = torch.softmax(outputs[0], dim=1)  # (T, 2)
            # Extract probability of class 1 (sepsis) for each timestep
            sepsis_probs = probs[:, 1].cpu().numpy()  # (T,)
            predictions_list.append(sepsis_probs)

    return predictions_list

def train_epoch_transformer(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    total_samples = 0

    for sequences, labels, attention_mask in train_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        attention_mask = attention_mask.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(sequences, attention_mask)

        # Reshape for loss calculation
        outputs_flat = outputs.reshape(-1, outputs.shape[-1])
        labels_flat = labels.reshape(-1)

        loss = criterion(outputs_flat, labels_flat)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics (only count non-padding positions)
        total_loss += loss.item() * (labels_flat != -1).sum().item()
        total_samples += (labels_flat != -1).sum().item()

    avg_loss = total_loss / total_samples if total_samples > 0 else 0
    return avg_loss

def compute_loss_transformer(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_samples = 0

    with torch.no_grad():
        for sequences, labels, attention_mask in data_loader:
            sequences = sequences.to(device)
            labels = labels.to(device)
            attention_mask = attention_mask.to(device)

            outputs = model(sequences, attention_mask)

            # Reshape for loss calculation
            outputs_flat = outputs.reshape(-1, outputs.shape[-1])
            labels_flat = labels.reshape(-1)

            loss = criterion(outputs_flat, labels_flat)

            total_loss += loss.item() * (labels_flat != -1).sum().item()
            total_samples += (labels_flat != -1).sum().item()

    avg_loss = total_loss / total_samples if total_samples > 0 else 0
    return avg_loss

def train_model_transformer(model, trn_loader, val_loader, criterion, optimizer, device,
                           num_epochs=20, checkpoint_path='best_transformer_model.pt',
                           val_dataset=None, utility_fn=patient_utility):
    trn_losses = []
    val_losses = []
    val_utilities = []
    optimal_thresholds = []

    best_val_utility = float('-inf')
    best_threshold = 0.05
    epochs_without_improvement = 0

    if val_dataset is None:
        raise ValueError("val_dataset must be provided for early stopping based on validation utility")

    for epoch in range(num_epochs):
        # Train for one epoch
        trn_loss = train_epoch_transformer(model, trn_loader, criterion, optimizer, device)

        # Use the same validation predictions for loss, threshold, and checkpoint selection.
        print(f"Epoch {epoch+1}/{num_epochs} - Generating validation predictions...")
        val_loss = compute_loss_transformer(model, val_loader, criterion, device)
        val_predictions = get_transformer_predictions(model, val_dataset, device)

        trn_losses.append(trn_loss)
        val_losses.append(val_loss)

        # Optimize threshold during validation
        print(f"Epoch {epoch+1}/{num_epochs} - Optimizing threshold on validation set...")
        threshold_results = optimize_threshold(
            predicted_probs_list=val_predictions,
            dataset=val_dataset,
            thresholds=np.linspace(0.01, 0.2, 20)
        )

        # Extract results
        epoch_best_threshold = threshold_results['best_threshold']
        epoch_best_utility = threshold_results['best_utility']

        optimal_thresholds.append(epoch_best_threshold)
        val_utilities.append(epoch_best_utility)

        # Log progress
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {trn_loss:.4f}, Val Loss: {val_loss:.4f}, " +
              f"Optimal Threshold: {epoch_best_threshold:.4f}, Val Utility: {epoch_best_utility:.4f}")

        if epoch_best_utility > best_val_utility:
            best_val_utility = epoch_best_utility
            best_threshold = epoch_best_threshold
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_utility': epoch_best_utility,
                'trn_loss': trn_loss,
                'optimal_threshold': epoch_best_threshold,
            }, checkpoint_path)
            print(f"  -> Checkpoint saved (best val utility: {epoch_best_utility:.4f}, threshold: {epoch_best_threshold:.4f})")

    # Load best model
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only = False)
    model.load_state_dict(checkpoint['model_state_dict'])
    best_threshold = checkpoint['optimal_threshold']
    print(f"\nLoaded best model from epoch {checkpoint['epoch']+1} with val utility {checkpoint['val_utility']:.4f} and threshold {best_threshold:.4f}")

    return trn_losses, val_losses, val_utilities, optimal_thresholds, best_threshold

def collator(batch):
    """
    Collate function that converts NumPy arrays to tensors
    and pads variable-length sequences.
    """
    sequences, labels = zip(*batch)

    # Convert NumPy arrays to PyTorch tensors
    sequences = [torch.as_tensor(x, dtype=torch.float32) for x in sequences]
    labels = [torch.as_tensor(y, dtype=torch.long) for y in labels]

    # Pad sequences
    sequences_padded = pad_sequence(
        sequences, batch_first=True, padding_value=0.0
    )
    labels_padded = pad_sequence(
        labels, batch_first=True, padding_value=-1
    )

    # Create attention mask (True for real data, False for padding)
    attention_mask = torch.ones(
        sequences_padded.shape[:2], dtype=torch.bool
    )
    for i, seq in enumerate(sequences):
        attention_mask[i, len(seq):] = False

    return sequences_padded, labels_padded, attention_mask

batch_size = 32
trn_loader = DataLoader(dataset_trn, batch_size=batch_size, shuffle=True, collate_fn=collator)
val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, collate_fn=collator)

Run the cell below to train the transformer.

In [ ]:
torch.manual_seed(RANDOM_SEED)
n_features = len(feature_names)
basic_model = Transformer(input_dim=n_features).to(device)
basic_criterion = nn.CrossEntropyLoss(ignore_index=-1)
basic_optimizer = optim.Adam(basic_model.parameters(), lr=0.0001)

num_epochs = 50
basic_trn_losses, basic_val_losses, basic_val_utilities, _, basic_threshold = train_model_transformer(
    basic_model, trn_loader, val_loader,
    basic_criterion, basic_optimizer, device, num_epochs,
    checkpoint_path="best_basic_transformer.pt",
    val_dataset=dataset_val,
)
basic_selected_epoch = int(np.argmax(basic_val_utilities)) + 1

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

epochs = range(1, num_epochs + 1)
ax1.plot(epochs, basic_trn_losses, label='Training loss')
ax1.plot(epochs, basic_val_losses, label='Validation loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training (basic transformer)')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, basic_val_utilities, label='Validation utility', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Utility')
ax2.set_title('Validation utility (with optimal threshold)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('4dii.pdf', bbox_inches = 'tight')

plt.show()

# The checkpoint was selected by validation utility.
selected_epoch = int(np.argmax(basic_val_utilities))
print(f"Selected epoch: {selected_epoch + 1}")
print(f"Selected-model validation utility: {basic_val_utilities[selected_epoch]:.4f}")
print(f"Selected-model validation loss: {basic_val_losses[selected_epoch]:.4f}")
print(f"Selected threshold: {basic_threshold:.4f}")

# 5 Hospital trial

Run the following cell to load in the trial dataset.

In [ ]:
dataset_trl = PatientDataset.from_file("trl.npz")
dataset_trl_raw = dataset_trl

dataset_trl = forward_fill_with_mean(dataset_trl)

## 5(a) Baseline trial utilities

In [ ]:
# TODO: 5(a) baseline trial utilities

...

## 5(b) Transformer trial utility

In [ ]:
@torch.no_grad()
def rolling_transformer_predictions(model, dataset, device, batch_size=128):
    """Evaluate rolling prefixes in batches; return hourly probabilities."""
    model.eval()
    records = [torch.as_tensor(x, dtype=torch.float32) for x, _ in dataset]
    prefixes = [(n, t) for n, x in enumerate(records) for t in range(1, len(x) + 1)]
    predictions = [np.empty(len(x)) for x in records]

    for start in trange(0, len(prefixes), batch_size, desc="Rolling evaluation"):
        batch = prefixes[start:start + batch_size]
        lengths = torch.tensor([t for n, t in batch], device=device)
        
        # Each batch row is a separate prefix; padding is hidden from attention.
        x = pad_sequence([records[n][:t] for n, t in batch], batch_first=True).to(device)
        attention_mask = torch.arange(x.shape[1], device=device)[None, :] < lengths[:, None]
        logits = model(x, attention_mask)[torch.arange(len(batch), device=device), lengths - 1]
        labels = torch.tensor([int(dataset[n][1][t - 1]) for n, t in batch], device=device)

        # Keep only each prefix's last output: the prediction for its final hour.
        probabilities = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        for (n, t), probability in zip(batch, probabilities):
            predictions[n][t - 1] = probability

    return predictions

In [ ]:
basic_trl_preds = rolling_transformer_predictions(basic_model, dataset_trl, device)
basic_trl_utility = compute_utility(basic_trl_preds, dataset_trl, threshold=basic_threshold)

print(f"Trial utility: {basic_trl_utility:.4f}")

# 6 Causal Self-Attention

## 6(b) Implement causal self-attention

In [ ]:
# TODO: 6(b) Implement causal self-attention

class CausalTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=3, dim_feedforward=512):
        """
        Args:
            input_dim: Number of input features (n_features)
            d_model: Dimension of the model
            nhead: Number of attention heads
            num_layers: Number of transformer encoder layers
            dim_feedforward: Dimension of feedforward network
            dropout: Dropout rate
        """
        super().__init__()

        # Input projection (no positional encoding)
        self.input_layer = nn.Linear(input_dim, d_model)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.0,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output layer 
        self.output_layer = nn.Linear(d_model, 2)

    def forward(self, x, attention_mask=None):
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, input_dim)
            attention_mask: Boolean tensor of shape (batch_size, seq_len);
                True for valid positions, False for padding

        Returns:
            outputs: Tensor of shape (batch_size, seq_len, 2) containing logits
        """
        padding_mask = ~attention_mask.bool() if attention_mask is not None else None
        ###
        # TODO: 6(b). Implement the forward pass with the causal mask. When calling 
        # self.transformer_encoder, pass in padding_mask to the src_key_padding_mask 
        # parameter and use the mask parameter to control which timesteps can 
        # attend to which other timesteps.
        ...
        return ...
        ###

## 6(c) Train the causal transformer

In [ ]:
torch.manual_seed(RANDOM_SEED)
n_features = len(feature_names)
causl_model = CausalTransformer(input_dim=n_features).to(device)
causl_criterion = nn.CrossEntropyLoss(ignore_index=-1)
causl_optimizer = optim.Adam(causl_model.parameters(), lr=0.0001)

num_epochs = 50
causl_trn_losses, causl_val_losses, causl_val_utilities, _, causl_threshold = train_model_transformer(
    causl_model, trn_loader, val_loader,
    causl_criterion, causl_optimizer, device, num_epochs,
    checkpoint_path="best_causl_transformer.pt",
    val_dataset=dataset_val,
)
causl_selected_epoch = int(np.argmax(causl_val_utilities)) + 1

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

epochs = range(1, num_epochs + 1)
ax1.plot(epochs, causl_trn_losses, label='Training loss',)
ax1.plot(epochs, causl_val_losses, label='Validation loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training (causal transformer)')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, causl_val_utilities, label='Validation utility', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Utility')
ax2.set_title('Validation utility (with optimal threshold)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('6c.pdf', bbox_inches = 'tight')
plt.show()

# The checkpoint was selected by validation utility.
selected_epoch = int(np.argmax(causl_val_utilities))
print(f"Selected epoch: {selected_epoch + 1}")
print(f"Selected-model validation utility: {causl_val_utilities[selected_epoch]:.4f}")
print(f"Selected-model validation loss: {causl_val_losses[selected_epoch]:.4f}")
print(f"Selected threshold: {causl_threshold:.4f}")

## 5(d) Causal transformer trial utility

In [ ]:
causl_trl_preds = rolling_transformer_predictions(causl_model, dataset_trl, device)
causl_trl_utility = compute_utility(causl_trl_preds, dataset_trl, threshold=causl_threshold)

print(f"Trial utility: {causl_trl_utility:.4f}")

# 7 Positional Encoding

## 7(a) Implement positional encoding

In [ ]:
# TODO: 7(a) Implement positional encoding.

class PositionalEncodingTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=3, dim_feedforward=512):
        """
        Args:
            input_dim: Number of input features (n_features)
            d_model: Dimension of the model
            nhead: Number of attention heads
            num_layers: Number of transformer encoder layers
            dim_feedforward: Dimension of feedforward network
            dropout: Dropout rate
        """
        super(PositionalEncodingTransformer, self).__init__()

        # Input layer
        self.input_layer = nn.Linear(input_dim, d_model)

        # For the sine/cosine positional encoding
        self.d_model = d_model

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.0,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output layer (binary classification at each time step)
        self.output_layer = nn.Linear(d_model, 2)

    def _get_positional_encoding(self, seq_len, device):
        """
        Generate sine/cosine positional encodings.
        """
        pe = torch.zeros(seq_len, self.d_model, device=device)
        position = torch.arange(0, seq_len, dtype=torch.float, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.d_model, 2, dtype=torch.float, device=device) *
                            (-np.log(10000.0) / self.d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0)  # (1, seq_len, d_model)

    def forward(self, x, attention_mask=None):
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, input_dim)
            attention_mask: Boolean tensor of shape (batch_size, seq_len), True for valid positions

        Returns:
            outputs: Tensor of shape (batch_size, seq_len, 2)
        """
        padding_mask = ~attention_mask.bool() if attention_mask is not None else None
        ###
        # TODO: 7(a). Implement the forward pass with the positional encoding. 
        # When calling self.transformer_encoder, pass in padding_mask to the 
        # src_key_padding_mask parameter and use the mask parameter to control which 
        # timesteps can attend to which other timesteps.
        ...
        return ...
        ###

## 7(b) Train the positional encoding transformer

In [ ]:
torch.manual_seed(RANDOM_SEED)
n_features = len(feature_names)
pstnl_model = PositionalEncodingTransformer(input_dim=n_features).to(device)
pstnl_criterion = nn.CrossEntropyLoss(ignore_index=-1)
pstnl_optimizer = optim.Adam(pstnl_model.parameters(), lr=0.0001)

num_epochs = 50
pstnl_trn_losses, pstnl_val_losses, pstnl_val_utilities, _, pstnl_threshold = train_model_transformer(
    pstnl_model, trn_loader, val_loader,
    pstnl_criterion, pstnl_optimizer, device, num_epochs,
    checkpoint_path="best_pstnl_transformer.pt",
    val_dataset=dataset_val,
)
pstnl_selected_epoch = int(np.argmax(pstnl_val_utilities)) + 1

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

epochs = range(1, num_epochs + 1)
ax1.plot(epochs, pstnl_trn_losses, label='Train loss')
ax1.plot(epochs, pstnl_val_losses, label='Validation loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training (positional encoding transformer)')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, pstnl_val_utilities, label='Validation utility', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Utility')
ax2.set_title('Validation utility (with optimal threshold)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('7b.pdf', bbox_inches = 'tight')
plt.show()

# The checkpoint was selected by validation utility.
selected_epoch = int(np.argmax(pstnl_val_utilities))
print(f"Selected epoch: {selected_epoch + 1}")
print(f"Selected-model validation utility: {pstnl_val_utilities[selected_epoch]:.4f}")
print(f"Selected-model validation loss: {pstnl_val_losses[selected_epoch]:.4f}")
print(f"Selected threshold: {pstnl_threshold:.4f}")

# 7(c) Positional encoding transformer trial utility

In [ ]:
pstnl_trl_preds = rolling_transformer_predictions(pstnl_model, dataset_trl, device)
pstnl_trl_utility = compute_utility(pstnl_trl_preds, dataset_trl, threshold=pstnl_threshold)

print(f"Trial utility: {pstnl_trl_utility:.4f}")

# 8 Model comparison

Run the following cell to load in the trial dataset.

In [ ]:
dataset_tst = PatientDataset.from_file("tst.npz")
dataset_tst = forward_fill_with_mean(dataset_tst)

## 8(a) Test utilities

In [ ]:
basic_tst_preds = rolling_transformer_predictions(basic_model, dataset_tst, device)
basic_tst_utility = compute_utility(basic_tst_preds, dataset_tst, threshold=basic_threshold)

causl_tst_preds = rolling_transformer_predictions(causl_model, dataset_tst, device)
causl_tst_utility = compute_utility(causl_tst_preds, dataset_tst, threshold=causl_threshold)

pstnl_tst_preds = rolling_transformer_predictions(pstnl_model, dataset_tst, device)
pstnl_tst_utility = compute_utility(pstnl_tst_preds, dataset_tst, threshold=pstnl_threshold)

print(f"Test utilities")
print(f"Basic transformer:\t{basic_tst_utility:.4f}")
print(f"Causal transformer:\t{causl_tst_utility:.4f}")
print(f"Positional encoding:\t{pstnl_tst_utility:.4f}")